# Repetition code Testing

In [6]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import Aer, execute
from qiskit.tools.visualization import circuit_drawer
from qiskit.extensions.simulator import *
import random

In [39]:
# Repetition code for 1 qubit, turns a|0> + b|1> into a|000> + b|111>
def rep_code(err_prob):
    q = QuantumRegister(3, name = 'q')
    c = ClassicalRegister(3, name = 'c')
    anc = QuantumRegister(2, name='a')
    out = ClassicalRegister(2, name='out')

    qc = QuantumCircuit(q, c, anc, out, name='rep_circuit')

    # q[0] is the initial logical qubit
    # spread the state across q[1] and q[2] as well, where q[1] and q[2]
    # are set to |0> initially
    qc.cx(q[0], q[1])
    qc.cx(q[0], q[2])

    if random.random() <= err_prob:
        i = random.randint(0, 2)
        qc.x(q[i])
        print(f'bit {i+1} flipped from noise')

    # Syndrome/measure the result
    qc.cx(q[0], anc[0])
    qc.cx(q[1], anc[0])
    qc.cx(q[0], anc[1])
    qc.cx(q[2], anc[1])

    qc.measure(anc, out)

    backend = Aer.get_backend('qasm_simulator')
    job= execute(qc, backend)
    res = job.result()
    print(res.get_counts(qc))

    errorbits = list(res.get_counts().keys())[0].split()[0]
    if errorbits=='10':
        print('error in the 3rd bit')
    elif errorbits=='01':
        print('error in the 2nd bit')
    elif errorbits=='11':
        print('error in the 1st bit')
    else:
        print('no error detected')
    


In [46]:
rep_code(0.5)

bit 2 flipped from noise
{'01 000': 1024}
error in the 2nd bit


In [54]:
def shor_code(err_prob):
    q = QuantumRegister(9)
    c = ClassicalRegister(9)
    qc = QuantumCircuit(q, c, name='shor_circuit')

    # Spread q[0] state across q[3] and q[6] - checks for phase erros
    qc.cx(q[0], q[3])
    qc.cx(q[0], q[6])
    for i in range(0,9,3): qc.h(q[i])

    for i in range(0,9,3):
        for j in range(1,3):
            qc.cx(q[i], q[i+j])
    
    # codestr =  qc.draw()
    # if random.random() < err_prob:
    #     i = random.choice(list(range(9)))
    #     if random.random()<0.5:
    #         print(f'Phase of {i} flipped')
    #         qc.h(q[i])
    #     else:
    #         print(f'Bit {i} flipped')
    #         qc.x(q[i])
    
    for i in range(0,9,3):
        for j in range(1,3):
            qc.cx(q[i], q[i+j])
    
    for i in range(0,9,3):
        qc.ccx(q[i+2], q[i+1], q[i])
        qc.h(q[i])
    
    qc.cx(q[0],q[3])
    qc.cx(q[0],q[6])
    qc.ccx(q[6], q[3], q[0])

    qc.measure(q, c)
    backend = Aer.get_backend('qasm_simulator')
    job = execute(qc, backend)
    res = job.result()

    # print('Measured result: ', res)
    print(res.get_counts(qc))

    return qc.draw()
    

shor_code(0.6)


Phase of 7 flipped
{'001000000': 519, '010000000': 505}


┌───┐                    ┌───┐┌───┐                        »
q325_0: ──■────■──┤ H ├──■────■────■────■──┤ X ├┤ H ├──■──────────────────■──»
          │    │  └───┘┌─┴─┐  │  ┌─┴─┐  │  └─┬─┘└───┘  │          ┌─┐     │  »
q325_1: ──┼────┼───────┤ X ├──┼──┤ X ├──┼────■─────────┼──────────┤M├─────┼──»
          │    │       └───┘┌─┴─┐└───┘┌─┴─┐  │         │          └╥┘┌─┐  │  »
q325_2: ──┼────┼────────────┤ X ├─────┤ X ├──■─────────┼───────────╫─┤M├──┼──»
        ┌─┴─┐  │  ┌───┐     └───┘     └───┘┌───┐┌───┐┌─┴─┐         ║ └╥┘  │  »
q325_3: ┤ X ├──┼──┤ H ├──■────■────■────■──┤ X ├┤ H ├┤ X ├─────────╫──╫───┼──»
        └───┘  │  └───┘┌─┴─┐  │  ┌─┴─┐  │  └─┬─┘└───┘└┬─┬┘         ║  ║   │  »
q325_4: ───────┼───────┤ X ├──┼──┤ X ├──┼────■────────┤M├──────────╫──╫───┼──»
               │       └───┘┌─┴─┐└───┘┌─┴─┐  │        └╥┘ ┌─┐      ║  ║   │  »
q325_5: ───────┼────────────┤ X ├─────┤ X ├──■─────────╫──┤M├──────╫──╫───┼──»
             ┌─┴─┐┌───┐     └───┘     └───┘     ┌───┐  ║  └╥┘┌───┐ ║  ║ ┌─┴─┐»
q325_6: ─────┤ X ├┤ H ├──■────■─────────■────■──┤ X ├──╫───╫─┤ H ├─╫──╫─┤ X ├»
             └───┘└───┘┌─┴─┐  │  ┌───┐┌─┴─┐  │  └─┬─┘  ║   ║ └┬─┬┘ ║  ║ └───┘»
q325_7: ───────────────┤ X ├──┼──┤ H ├┤ X ├──┼────■────╫───╫──┤M├──╫──╫──────»
                       └───┘┌─┴─┐└───┘└───┘┌─┴─┐  │    ║   ║  └╥┘  ║  ║  ┌─┐ »
q325_8: ────────────────────┤ X ├──────────┤ X ├──■────╫───╫───╫───╫──╫──┤M├─»
                            └───┘          └───┘       ║   ║   ║   ║  ║  └╥┘ »
 c10: 9/═══════════════════════════════════════════════╩═══╩═══╩═══╩══╩═══╩══»
                                                       4   5   7   1  2   8  »
«        ┌───┐┌─┐      
«q325_0: ┤ X ├┤M├──────
«        └─┬─┘└╥┘      
«q325_1: ──┼───╫───────
«          │   ║       
«q325_2: ──┼───╫───────
«          │   ║ ┌─┐   
«q325_3: ──■───╫─┤M├───
«          │   ║ └╥┘   
«q325_4: ──┼───╫──╫────
«          │   ║  ║    
«q325_5: ──┼───╫──╫────
«          │   ║  ║ ┌─┐
«q325_6: ──■───╫──╫─┤M├
«              ║  ║ └╥┘
«q325_7: ──────╫──╫──╫─
«              ║  ║  ║ 
«q325_8: ──────╫──╫──╫─
«              ║  ║  ║ 
« c10: 9/══════╩══╩══╩═
«              0  3  6